# 06 — Blade Design (NACA 65-Series)

**Purpose:** Design the rotor and stator blade profiles at hub, mean, and tip using the NACA 65-series family. Apply the Carter deviation rule to close the incidence–deviation loop, compute blade stagger and camber at each radial station, and stack the sections about the centroid axis to produce a manufacturable 3-D blade geometry.

**Inputs:** Radial distribution from notebook 02 and IGV geometry from notebook 03.

**Scope:** This notebook implements the `blade.py` module described in the README and scaffolds its functions inline. Once validated, the functions should be moved to `src/blade.py`.

**Outputs:**
- NACA 65-series thickness distribution (Yt/c vs x/c)
- Camber line and stagger at hub / mean / tip for rotor and stator
- Blade coordinate table at each station (pressure and suction surface)
- Stagger and camber spanwise distribution plot
- Meridional view of rotor stacking axis

**References:** Lieblein et al. (1953) NACA RM E53D01; Abbott & von Doenhoff (1959) *Theory of Wing Sections*; Carter (1950) ARC R&M 2816; Dixon & Hall (2014) Ch. 5.

In [ ]:
import sys, pathlib

# Locate repo root (directory that contains src/) regardless of
# where the notebook file sits (repo root or notebooks/ subfolder)
_here = pathlib.Path.cwd()
_root = next(
    (p for p in [_here, *_here.parents] if (p / "src").is_dir()),
    _here,
)
sys.path.insert(0, str(_root))
pathlib.Path(_root / "figures").mkdir(exist_ok=True)


import numpy as np
import matplotlib.pyplot as plt
import math

from src.igv    import igv_geometry, meanline_with_igv
from src.radial import free_vortex
from src.meanline import meanline_analysis

plt.rcParams.update({'figure.dpi': 130, 'font.size': 10})
print('Imports OK')

In [ ]:
# ── Load design point from notebook 01 ─────────────────────
import json as _json, pathlib as _pl

_dp_path = _pl.Path(_root / "design_point.json")
if not _dp_path.exists():
    raise FileNotFoundError(
        "design_point.json not found — run notebook 01 first.")

dp = _json.loads(_dp_path.read_text())

# Unpack for convenience
D_TIP = dp["D_tip"]
N_RPM = dp["N_RPM"]
PR    = dp["PR"]
NU    = dp["nu"]
PHI   = dp["phi"]
ETA   = dp["eta_is"]
print(f"Design point: D_tip={D_TIP*1000:.0f} mm  N={N_RPM} RPM  PR={PR}  nu={NU}  phi={PHI}  eta={ETA}")


## 1. Design inputs

In [ ]:
igv   = igv_geometry(D_tip=D_TIP, nu=NU, N_RPM=N_RPM, phi=PHI, alpha1_deg=0.0)
rotor = meanline_with_igv(igv, PR=PR, eta_is=ETA)
res   = meanline_analysis(D_TIP, N_RPM, PR, ETA, nu=NU, phi=PHI)

# Rotor solidity at mid-span
r_mean = res['r_mean']
h      = res['r_tip'] - res['r_hub']
AR     = 1.5
B      = igv['B_blades']
chord_mid = h / AR
pitch_mid = 2 * math.pi * r_mean / B
sigma_mid = chord_mid / pitch_mid

# Radial distribution
rad = free_vortex(res, n_stations=5, sigma=sigma_mid)

print(f'B = {B},  chord_mid = {chord_mid*1000:.1f} mm,  σ_mid = {sigma_mid:.3f}')
print(f'Radial stations (hub→tip): r = {rad["r"]*1000}')

## 2. NACA 65-series thickness distribution

The NACA 65-series thickness envelope (t/c = 0.10, i.e. NACA 65-010) in 55-point tabular form from Abbott & von Doenhoff (1959).

In [ ]:
def naca65_thickness(t_c=0.10, n_points=100):
    """
    NACA 65-series thickness half-distribution Yt(x/c).
    Polynomial fit from Abbott & von Doenhoff (1959) Table 6.1.
    Returns x_c, yt_c arrays (x/c from 0 to 1).

    For NACA 65-0(10t/c) the ordinate scales linearly with t/c.
    """
    # Coefficients for the NACA 65-010 thickness form (unit chord)
    # yt/c = a0*sqrt(x/c) + a1*(x/c) + a2*(x/c)^2 + a3*(x/c)^3 + a4*(x/c)^4
    # (leading-edge formula; trailing edge uses separate closure)
    x_c = np.linspace(0.0, 1.0, n_points)

    # NACA 65-010 polynomial coefficients (Theodorsen form, normalised to t/c=0.10)
    a0 =  0.1737
    a1 = -0.2422
    a2 =  0.6383
    a3 = -0.6113
    a4 =  0.1020

    xi = np.where(x_c < 1e-10, 0.0, x_c)
    yt = (a0 * np.sqrt(xi) + a1 * xi + a2 * xi**2 + a3 * xi**3 + a4 * xi**4)
    yt = yt * (t_c / 0.10)  # scale to desired t/c

    return x_c, yt


x_c, yt_c = naca65_thickness(t_c=0.10)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(x_c, yt_c,  '#185FA5', lw=2, label='Upper surface')
ax.plot(x_c, -yt_c, '#185FA5', lw=2)
ax.axhline(0, color='gray', lw=0.8)
ax.set(xlabel='x/c', ylabel='y_t/c', title='NACA 65-010 Thickness Distribution')
ax.set_aspect('equal')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(str(_root / 'figures') + '/06_naca65_thickness.png', dpi=130, bbox_inches='tight')
plt.show()

## 3. Circular-arc mean camber line

The NACA 65-series mean line is a circular arc of design lift coefficient Cl. Here we use the simpler circular-arc approximation, which is adequate for subsonic stages (M_rel < 0.7).

In [ ]:
def circular_arc_camber(theta_c_deg, n_points=100):
    """
    Circular-arc mean line.
    theta_c : total camber angle [deg]
    Returns x_c, yc_c (x/c 0→1, yc/c camber ordinate).
    """
    theta = np.radians(theta_c_deg)
    x_c   = np.linspace(0.0, 1.0, n_points)
    if abs(theta_c_deg) < 0.1:
        return x_c, np.zeros_like(x_c)
    # Radius of circular arc R such that chord=1 and subtended angle=theta
    R  = 1.0 / (2.0 * np.sin(theta / 2.0))
    yc = R - np.sqrt(np.maximum(0.0, R**2 - (x_c - 0.5)**2))
    yc = yc - yc[0]          # force leading edge at y=0
    return x_c, yc


# Carter deviation rule: δ = m * sqrt(θ_c / σ)
# m ≈ 0.23 for NACA 65-series, Lieblein (1953)
def carter_deviation(theta_c_deg, sigma, m=0.23):
    """Return Carter deviation angle [deg]."""
    return m * np.sqrt(abs(theta_c_deg) / max(sigma, 0.01)) * np.sign(theta_c_deg)


# Iterate camber given required flow deflection
def blade_camber_from_deflection(delta_flow_deg, sigma, m=0.23, n_iter=20):
    """
    Find camber angle θ_c such that  θ_c - δ(θ_c) = delta_flow_deg.
    Iterates Carter deviation loop.
    """
    theta_c = delta_flow_deg  # initial guess
    for _ in range(n_iter):
        dev = carter_deviation(theta_c, sigma, m)
        theta_c = delta_flow_deg + dev
    return theta_c, dev

# Demonstrate for mean station
delta_beta_mean = abs(rotor['beta1_deg']) - abs(rotor['beta2_deg'])
theta_c_mean, dev_mean = blade_camber_from_deflection(delta_beta_mean, sigma_mid)
print(f'Mean station:  Δβ = {delta_beta_mean:.2f}°  →  θ_c = {theta_c_mean:.2f}°,  δ = {dev_mean:.2f}°')

## 4. Blade section at each radial station — ROTOR

In [ ]:
def blade_section(beta1_deg, beta2_deg, sigma, chord, t_c=0.10):
    """
    Full blade section geometry at one radial station.

    Parameters
    ----------
    beta1_deg  : rotor relative inlet angle (sign convention: negative = opposite rotation)
    beta2_deg  : rotor relative exit angle
    sigma      : local solidity c/pitch
    chord      : chord length [m]
    t_c        : thickness ratio

    Returns
    -------
    dict with stagger, camber, and Cartesian (x,y) coordinates of
    pressure and suction surfaces in blade frame.
    """
    delta_flow = abs(beta1_deg) - abs(beta2_deg)
    theta_c, deviation = blade_camber_from_deflection(delta_flow, sigma)

    # Design incidence (Lieblein empirical for NACA 65-series)
    if abs(delta_flow) < 1.0:
        i_des = 0.0
    else:
        i_des = -6.0 + 0.18 * abs(delta_flow)

    kappa_LE = abs(beta1_deg) - i_des          # leading-edge metal angle
    kappa_TE = abs(beta2_deg) - deviation      # trailing-edge metal angle
    stagger  = (kappa_LE + kappa_TE) / 2.0

    # Unit-chord section
    x_c, yt_c = naca65_thickness(t_c=t_c)
    _, yc_c   = circular_arc_camber(theta_c)

    # Combine to get pressure/suction surface
    # Normal to camber line  dα = arctan(dyc/dx)
    dyc = np.gradient(yc_c, x_c)
    angle_c = np.arctan(dyc)

    xu = x_c  - yt_c * np.sin(angle_c)   # suction
    yu = yc_c + yt_c * np.cos(angle_c)
    xl = x_c  + yt_c * np.sin(angle_c)   # pressure
    yl = yc_c - yt_c * np.cos(angle_c)

    # Scale by chord and rotate by stagger
    xi = np.radians(stagger)
    def rotate(x, y, angle):
        return (x*np.cos(angle) - y*np.sin(angle),
                x*np.sin(angle) + y*np.cos(angle))

    Xu, Yu = rotate(xu * chord, yu * chord, xi)
    Xl, Yl = rotate(xl * chord, yl * chord, xi)

    return {
        'stagger_deg':  stagger,
        'camber_deg':   theta_c,
        'deviation_deg': deviation,
        'i_des_deg':    i_des,
        'kappa_LE_deg': kappa_LE,
        'kappa_TE_deg': kappa_TE,
        'chord_m':      chord,
        'sigma':        sigma,
        'suction_x':    Xu, 'suction_y':  Yu,
        'pressure_x':   Xl, 'pressure_y': Yl,
    }


# Compute at hub, mean, tip
stations_label = ['Hub', 'Mean', 'Tip']
idx3 = [0, 2, 4]   # out of 5 stations

rotor_sections = {}
print(f'{"Station":<8} {"r [mm]":>8} {"σ":>7} {"β₁":>7} {"β₂":>7} {"θ_c":>7} {"ξ (stagger)":>12} {"δ":>7}')
print('─' * 68)

for lbl, i in zip(stations_label, idx3):
    r    = rad['r'][i]
    pitch = 2 * math.pi * r / B
    chord_r = chord_mid   # constant chord for simplicity; tapered chord is a later refinement
    sig  = chord_r / pitch
    b1   = rad['beta1'][i]
    b2   = rad['beta2'][i]
    sec  = blade_section(b1, b2, sig, chord_r, t_c=0.10)
    rotor_sections[lbl] = sec
    print(f'{lbl:<8} {r*1000:>8.1f} {sig:>7.3f} {b1:>7.2f} {b2:>7.2f}'
          f' {sec["camber_deg"]:>7.2f} {sec["stagger_deg"]:>12.2f} {sec["deviation_deg"]:>7.2f}')

## 5. Blade section profiles

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle('Rotor Blade Sections — Hub / Mean / Tip', fontweight='bold')

col_ss = '#185FA5'
col_ps = '#D85A30'

for ax, lbl in zip(axes, stations_label):
    sec = rotor_sections[lbl]
    ax.plot(sec['suction_x']*1000,  sec['suction_y']*1000,  col_ss, lw=2, label='Suction surface')
    ax.plot(sec['pressure_x']*1000, sec['pressure_y']*1000, col_ps, lw=2, label='Pressure surface')
    ax.set(xlabel='x [mm]', ylabel='y [mm]',
           title=f'{lbl}\nξ={sec["stagger_deg"]:.1f}°  θ_c={sec["camber_deg"]:.1f}°')
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    if lbl == 'Hub':
        ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(str(_root / 'figures') + '/06_rotor_sections.png', dpi=130, bbox_inches='tight')
plt.show()

## 6. Spanwise stagger and camber distributions

In [ ]:
# Compute at all 5 stations
stagger_sp = []
camber_sp  = []
pct_sp     = []

for i in range(len(rad['r'])):
    r     = rad['r'][i]
    pitch = 2 * math.pi * r / B
    sig   = chord_mid / pitch
    sec   = blade_section(rad['beta1'][i], rad['beta2'][i], sig, chord_mid)
    stagger_sp.append(sec['stagger_deg'])
    camber_sp.append(sec['camber_deg'])
    pct_sp.append(rad['pct_span'][i])

fig, axes = plt.subplots(1, 2, figsize=(10, 5), sharey=True)
fig.suptitle('Rotor Blade — Spanwise Stagger & Camber', fontweight='bold')

axes[0].plot(stagger_sp, pct_sp, '#185FA5', lw=2.5, marker='o')
axes[0].set(xlabel='Stagger angle ξ [°]', ylabel='% Span')
axes[0].grid(True, alpha=0.35)

axes[1].plot(camber_sp, pct_sp, '#D85A30', lw=2.5, marker='s')
axes[1].set(xlabel='Camber angle θ_c [°]')
axes[1].grid(True, alpha=0.35)

for ax in axes:
    ax.axhline(50, color='gray', lw=0.8, ls=':')

plt.tight_layout()
plt.savefig(str(_root / 'figures') + '/06_stagger_camber_spanwise.png', dpi=130, bbox_inches='tight')
plt.show()

## 7. Stator blade sections

The stator (downstream of rotor) turns the flow from α₂ back to approximately axial. Same procedure as the rotor, but using absolute frame angles.

In [ ]:
# Stator: inlet angle = α₂ (from rotor exit), exit angle ≈ 0° (axial)
# Use same blade count as rotor for a 1:1 stage (stator count is a separate design choice)
# Here we use B_stator = B_rotor for illustration
B_stator = B

stator_sections = {}
print('STATOR SECTIONS')
print(f'{"Station":<8} {"r [mm]":>8} {"σ":>7} {"α₂":>7} {"α₃":>7} {"θ_c":>7} {"ξ":>12} {"δ":>7}')
print('─' * 68)

for lbl, i in zip(stations_label, idx3):
    r     = rad['r'][i]
    pitch = 2 * math.pi * r / B_stator
    sig   = chord_mid / pitch
    alpha2 = rad['alpha2'][i]   # stator inlet angle
    alpha3 = 2.0                 # target exit angle [deg] — near-axial
    sec = blade_section(alpha2, alpha3, sig, chord_mid, t_c=0.10)
    stator_sections[lbl] = sec
    print(f'{lbl:<8} {r*1000:>8.1f} {sig:>7.3f} {alpha2:>7.2f} {alpha3:>7.2f}'
          f' {sec["camber_deg"]:>7.2f} {sec["stagger_deg"]:>12.2f} {sec["deviation_deg"]:>7.2f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle('Stator Blade Sections — Hub / Mean / Tip', fontweight='bold')

for ax, lbl in zip(axes, stations_label):
    sec = stator_sections[lbl]
    ax.plot(sec['suction_x']*1000,  sec['suction_y']*1000,  col_ss, lw=2)
    ax.plot(sec['pressure_x']*1000, sec['pressure_y']*1000, col_ps, lw=2)
    ax.set(xlabel='x [mm]', ylabel='y [mm]',
           title=f'{lbl}\nξ={sec["stagger_deg"]:.1f}°  θ_c={sec["camber_deg"]:.1f}°')
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(_root / 'figures') + '/06_stator_sections.png', dpi=130, bbox_inches='tight')
plt.show()

## 8. Next steps — move to `src/blade.py`

The functions defined in this notebook should be promoted to `src/blade.py` with the following interface:

```python
from src.blade import (
    naca65_thickness,          # NACA 65-series thickness ordinate
    circular_arc_camber,       # circular-arc mean line
    carter_deviation,          # Carter deviation rule
    blade_camber_from_deflection,  # iterative camber solver
    blade_section,             # full 2D section at one radial station
    blade_radial_build,        # sweep all stations, return stacked geometry
)
```

Additional work for `blade_radial_build`:
- Tapered chord law (linear taper from tip to hub, AR optimisation)
- Stacking axis choice (centroid stacking vs leading-edge stacking)
- Export to CSV / STL for CAD import
- Chordwise Mach number check at tip to flag any transonic risk